# Dynamic Feed — keyless live data via a remote MCP server

[Dynamic Feed](https://dynamicfeed.ai) is a keyless remote (Streamable HTTP) MCP server that gives a LlamaIndex agent live, current data its model was never trained on — weather, CVEs, software versions, hazards, satellites, macro and more (87 tools across 19 verticals). Every datapoint is Ed25519-signed and carries a provenance envelope (source, observed-at, freshness).

This notebook uses the existing `llama-index-tools-mcp` integration — no new package, no API key.

In [ ]:
%pip install llama-index-tools-mcp llama-index-llms-openai

In [ ]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

# keyless remote MCP endpoint
mcp_client = BasicMCPClient("https://dynamicfeed.ai/mcp")
tools = await McpToolSpec(client=mcp_client).to_tool_list_async()
print(f"loaded {len(tools)} keyless Dynamic Feed tools")

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

agent = FunctionAgent(tools=tools, llm=OpenAI(model="gpt-4o"))
resp = await agent.run("What is the latest stable Python, and is lodash 4.17.10 vulnerable?")
print(resp)

Each response is **Ed25519-signed** and verifiable against `https://dynamicfeed.ai/.well-known/keys`. The signature is proof-of-existence and tamper-evidence — it proves Dynamic Feed reported a value at a time and the bytes are unaltered. It is **not** a claim the value is objectively true; your agent still applies judgment.